# modify fits header

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [16]:
#import sys
#!pip install photutils==1.6

In [30]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        #print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module version_information is installed
The version_information extension is already loaded. To reload it, use:
  %reload_ext version_information
This notebook was generated at 2023-02-22 02:23:42 (대한민국 표준시 = GMT+0900) 
0 Python     3.9.7 64bit [MSC v.1916 64 bit (AMD64)]
1 IPython    7.31.1
2 OS         Windows 10 10.0.22621 SP0
3 numpy      1.21.5
4 pandas     1.4.1
5 matplotlib 3.5.1
6 scipy      1.7.3
7 astropy    5.0
8 photutils  1.6.0
9 ccdproc    2.3.1
10 version_information 1.0.4


### import modules

In [31]:
from glob import glob
from pathlib import Path, PosixPath, WindowsPath
import os
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.stats import sigma_clip
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.nddata import CCDData, Cutout2D
from astropy.stats import mad_std
from astropy.table import Table
from astropy.time import Time
from astropy.visualization import ImageNormalize, ZScaleInterval
from astropy.wcs import WCS, Wcsprm
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

import astro_utilities
import Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

In [32]:
checkKEYs = ["OBJECT", "TELESCOP", "OPTIC", "CCDNAME", 
            "GAIN", "EGAIN", "RDNOISE", "FOCALLEN", "PIXSCALE",
            "XBINNING", "YBINNING", "FLIPSTAT"]

In [33]:
#%%
BASEDIR = Path(r"r:\CCD_obs") 
BASEDIR = Path( BASEDIR/ astro_utilities.CCD_NEW_dir)

In [34]:
DOINGDIRs = sorted(Python_utilities.getFullnameListOfallsubDirs(BASEDIR))
#print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

len(DOINGDIRs):  60


In [35]:
for fpath in DOINGDIRs[:2] :
    #fpath = Path(DOINGDIRs[0])
    fpath = Path(fpath)
    #print(f"Starting: {str(fpath.parts[-1])}")
    #save_fpath = fpath/f"summary_{fpath.parts[-1]}.csv"

    try: 
        summary = yfu.make_summary(fpath/"*.fit*",
                    #output = save_fpath,
                    verbose = False
                    )
        print("summary", summary)
        print("len(summary)", len(summary))
    except:
        pass

summary None
summary                                                 file  filesize  SIMPLE  \
0  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   
1  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   
2  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   
3  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   
4  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   
5  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   
6  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   
7  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   
8  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   
9  r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT...  21427200    True   

   BITPIX  NAXIS  NAXIS1  NAXIS2  BZERO  EXTEND IMAGETYP  ...   OBJCTDEC  \
0      16      2    4008    2672  32768    True    LIGHT  ...  +10 26 06   
1      16   

In [36]:
for _, row in summary.iterrows():
    # 파일명 출력
    print (row["file"])
    fpath = Path(row["file"])

r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT_-_2023-02-20_-_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-_-\IC 2167_-LIGHT_H_2023-02-20_23-22-05_900.00_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-19.98_1x1.fits
r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT_-_2023-02-20_-_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-_-\IC 2167_-LIGHT_H_2023-02-20_23-37-58_900.00_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-20.42_1x1.fits
r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT_-_2023-02-20_-_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-_-\IC 2167_-LIGHT_H_2023-02-20_23-54-10_900.00_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-20.42_1x1.fits
r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT_-_2023-02-20_-_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-_-\IC 2167_-LIGHT_L_2023-02-20_22-09-00_600.00_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-20.42_1x1.fits
r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT_-_2023-02-20_-_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-_-\IC 2167_-LIGHT_L_2023-02-20_22-19-55_600.00_FSQ106ED-x72_SBIG

In [37]:
foldername_el = fpath.parts[-2].split('_')
print("foldername_el", foldername_el)
object_name = foldername_el[0]
optic_name = foldername_el[5]
ccd_name = foldername_el[6]
print("object_name", object_name)
print("ccd_name", ccd_name)

new_fpath = Path(f"{fpath.parents[0]}/{fpath.stem}_new.fit")

hdul = fits.open(str(fpath))
print(fpath)

foldername_el ['IC 2167', 'LIGHT', '-', '2023-02-20', '-', 'FSQ106ED-x72', 'SBIG STL-11000 3 CCD Camera', '-', '-']
object_name IC 2167
ccd_name SBIG STL-11000 3 CCD Camera
r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT_-_2023-02-20_-_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-_-\IC 2167_-LIGHT_L_2023-02-20_23-04-13_600.00_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-20.42_1x1_new.fit


In [38]:
for checkKEY in checkKEYs: 
    if not checkKEY in hdul[0].header :
        hdul[0].header.append(checkKEY, 
                        '', 
                        f"The keyword '{checkKEY}' is added") 
        print(f"The keyword '{checkKEY}' is added...")

The keyword 'OPTIC' is added...
The keyword 'GAIN' is added...
The keyword 'RDNOISE' is added...
The keyword 'PIXSCALE' is added...
The keyword 'FLIPSTAT' is added...


In [39]:
try : 
    if 'qsi' in hdul[0].header['INSTRUME'].lower() :     
        CCDNAME = 'QSI683ws'
    elif  'st-8300' in hdul[0].header['INSTRUME'].lower() : 
        CCDNAME = 'ST-8300M'
    elif  'stf-8300' in hdul[0].header['INSTRUME'].lower() : 
        CCDNAME = 'STF-8300M'
    elif  '11000' in hdul[0].header['INSTRUME'].lower() : 
        CCDNAME = 'STL-11000M'
    elif  '16803' in hdul[0].header['INSTRUME'].lower() : 
        CCDNAME = 'STX-16803'
except :
        CCDNAME = ccd_name
print("CCDNAME", CCDNAME)

if object_name != hdul[0].header["OBJECT"] : 
    hdul[0].header["OBJECT"] = object_name.upper
    print(f"The 'OBJECT' is set {object_name}")

if hdul[0].header["CCDNAME"] != CCDNAME : 
    hdul[0].header["CCDNAME"] = CCDNAME.upper
    print(f"The 'CCDNAME' is set {CCDNAME}")

if optic_name != hdul[0].header["OPTIC"] : 
    hdul[0].header["OPTIC"] = optic_name
    print(f"The 'OPTIC' is set {optic_name}")

hdul[0].header['GAIN'] = astro_utilities.GAINDIC[CCDNAME]
hdul[0].header['EGAIN'] = astro_utilities.GAINDIC[CCDNAME]
hdul[0].header['RDNOISE'] = astro_utilities.RDNOISEDIC[CCDNAME]

CCDNAME STL-11000M
The 'OPTIC' is set FSQ106ED-x72


In [40]:
for checkKEY in checkKEYs: 
    print(f"{checkKEY}: ", hdul[0].header[checkKEY])


OBJECT:  IC 2167
TELESCOP:  FSQ106ED-x72
OPTIC:  FSQ106ED-x72
CCDNAME:  STL-11000M
GAIN:  0.8
EGAIN:  0.8
RDNOISE:  9.6
FOCALLEN:  381.0
PIXSCALE:  None
XBINNING:  1
YBINNING:  1
FLIPSTAT:  None


In [41]:
hdul[0].header['GAIN'] = astro_utilities.GAINDIC[CCDNAME]
hdul[0].header['EGAIN'] = astro_utilities.GAINDIC[CCDNAME]
hdul[0].header['RDNOISE'] = astro_utilities.RDNOISEDIC[CCDNAME]
print(fpath)



r:\CCD_obs\CCD_new_files\13UD70P\IC 2167_LIGHT_-_2023-02-20_-_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-_-\IC 2167_-LIGHT_L_2023-02-20_23-04-13_600.00_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-20.42_1x1_new.fit


In [29]:
print(fpath)

hdul.writeto(str(fpath), overwrite=True)

PermissionError: [WinError 32] 다른 프로세스가 파일을 사용 중이기 때문에 프로세스가 액세스 할 수 없습니다: 'r:\\CCD_obs\\CCD_new_files\\13UD70P\\IC 2167_LIGHT_-_2023-02-20_-_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-_-\\IC 2167_-LIGHT_L_2023-02-20_23-04-13_600.00_FSQ106ED-x72_SBIG STL-11000 3 CCD Camera_-20.42_1x1_new.fit'